In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("all_subjects_2020_final.csv")

In [ ]:
df.shape

(125737, 13)

In [ ]:
#cleaning texts
import re

def clean_text(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove all non-alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
# Severity Creation

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
depression_anchors = [
    "I feel hopeless and empty all the time.",
    "I feel worthless, like nothing I do matters.",
    "I've lost interest in everything I used to enjoy.",
    "I'm so exhausted and I can't cope anymore.",
    "I feel completely alone and numb inside.",
    "I hate myself and I don't know why I keep going.",
]

suicide_anchors = [
    "I don't want to live anymore.",
    "I've been thinking about ending my life.",
    "I wish I could just disappear forever.",
    "I have a plan to kill myself.",
    "Sometimes I think everyone would be better off without me.",
    "I want to end it all.",
]

neutral_anchors = [
    "Today was a pretty normal day at work.",
    "I'm looking forward to the weekend with friends.",
    "Just watched a great movie last night.",
    "Trying out a new recipe for dinner tonight.",
]


# Initializing of Embedding model

model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts, batch_size=256):
    return model.encode(
        list(texts), batch_size=batch_size, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True  # normalized -> dot product == cosine sim
    )

print("Embedding anchors...")
dep_anchor_emb = embed(depression_anchors)
suicide_anchor_emb = embed(suicide_anchors)
neutral_anchor_emb = embed(neutral_anchors)

print("Embedding posts (this is the slow part on CPU -- use a GPU runtime if you can)...")
post_emb = embed(df["clean_text"].tolist())


# Creating Similarity scores: max cosine similarity to each anchor set
#    (we have created max, not mean -- a post only needs to strongly match ONE severity
#    signal to count, same logic as the original phrase-matching version)

def max_cosine_sim(post_embeddings, anchor_embeddings):
    sims = post_embeddings @ anchor_embeddings.T   # (n_posts, n_anchors), are already normalized
    return sims.max(axis=1)

df["depression_sim"] = max_cosine_sim(post_emb, dep_anchor_emb)
df["suicide_sim"] = max_cosine_sim(post_emb, suicide_anchor_emb)
df["neutral_sim"] = max_cosine_sim(post_emb, neutral_anchor_emb)


# Creating Composite severity score
df["raw_severity_score"] = (
      0.45 * df["depression_sim"]
    + 0.45 * df["suicide_sim"]
    - 0.10 * df["neutral_sim"]
)

scaler = MinMaxScaler()
df["severity_score"] = scaler.fit_transform(df[["raw_severity_score"]])


# Creating Rule-based severity labels using similarity thresholds

SUICIDE_THRESH = 0.55
DEPRESSION_THRESH = 0.50

def severity_label(row):
    if row["suicide_sim"] >= SUICIDE_THRESH:
        return 2
    elif row["depression_sim"] >= DEPRESSION_THRESH:
        return 1
    else:
        return 0

df["severity"] = df.apply(severity_label, axis=1)

print("\nSimilarity score distributions:")
print(df[["depression_sim", "suicide_sim", "neutral_sim"]].describe())
print("\nSeverity label counts:")
print(df["severity"].value_counts().sort_index())

df.to_pickle("df_embedding_severity.pkl")
df.to_csv("all_subjects_2020_final.csv", index=False)
print("\nSaved.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding anchors...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding posts (this is the slow part on CPU -- use a GPU runtime if you can)...


Batches:   0%|          | 0/492 [00:00<?, ?it/s]


Similarity score distributions:
       depression_sim    suicide_sim    neutral_sim
count   125737.000000  125737.000000  125737.000000
mean         0.119914       0.137212       0.136649
std          0.076820       0.077670       0.079468
min         -0.119058      -0.134220      -0.109417
25%          0.066330       0.083430       0.083082
50%          0.111912       0.132547       0.129987
75%          0.162959       0.184429       0.179066
max          0.644425       0.759280       0.730438

Severity label counts:
severity
0    125677
1        38
2        22
Name: count, dtype: int64

Saved.
